# Notebook 1 — Read & Join the Tables

**Goal:** load every Olist table from the database, understand it on its own,
then aggregate and join everything into **one ML table = one row per order**.

Adjust `DB_PATH` and the table names in the CONFIG cell below to match what you
used in Task 1 (SQLite file created from the Kaggle Olist CSVs).

In [1]:
# ---- CONFIG ----
import sqlite3
import pandas as pd
import numpy as np
import os

DB_PATH = "olist.db"          # <-- path to the SQLite DB you built in Task 1
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TABLES = {
    "orders": "olist_orders_dataset",
    "customers": "olist_customers_dataset",
    "order_items": "olist_order_items_dataset",
    "order_payments": "olist_order_payments_dataset",
    "order_reviews": "olist_order_reviews_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "category_translation": "product_category_name_translation",
}

conn = sqlite3.connect(DB_PATH)
# quick sanity check: list actual tables in the DB and compare to TABLES above
existing = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(existing)

                                name
0            olist_customers_dataset
1          olist_geolocation_dataset
2               olist_orders_dataset
3          olist_order_items_dataset
4       olist_order_payments_dataset
5        olist_order_reviews_dataset
6             olist_products_dataset
7              olist_sellers_dataset
8  product_category_name_translation


In [2]:
# ---- Load every table ----
dfs = {}
for key, table_name in TABLES.items():
    dfs[key] = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    print(f"{key:22s} -> {table_name:35s} shape={dfs[key].shape}")

orders                 -> olist_orders_dataset                shape=(99441, 8)
customers              -> olist_customers_dataset             shape=(99441, 5)
order_items            -> olist_order_items_dataset           shape=(112650, 7)
order_payments         -> olist_order_payments_dataset        shape=(103886, 5)
order_reviews          -> olist_order_reviews_dataset         shape=(99224, 7)
products               -> olist_products_dataset              shape=(32951, 9)
sellers                -> olist_sellers_dataset               shape=(3095, 4)
category_translation   -> product_category_name_translation   shape=(71, 2)


## Look at each table on its own
For every table: row count, key columns, duplicates, and what one row actually
represents. This is the part you should NOT skip — it's what tells you how to
join correctly in the next step.

In [3]:
for key, df in dfs.items():
    print("="*70)
    print(key)
    print(df.dtypes)
    print("n_rows:", len(df), "| n_duplicated_rows:", df.duplicated().sum())
    display(df.head(3))

orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
n_rows: 99441 | n_duplicated_rows: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
n_rows: 99441 | n_duplicated_rows: 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
n_rows: 112650 | n_duplicated_rows: 0


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


sellers
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object
n_rows: 3095 | n_duplicated_rows: 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


category_translation
product_category_name            str
product_category_name_english    str
dtype: object
n_rows: 71 | n_duplicated_rows: 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [4]:
# ---- Check keys & duplicates that matter for joining ----
orders = dfs["orders"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]

print("orders: order_id unique?      ", orders['order_id'].is_unique)
print("order_items: (order_id) unique?", order_items['order_id'].is_unique,
      "-> expected FALSE, many rows per order")
print("order_payments: (order_id) unique?", order_payments['order_id'].is_unique,
      "-> expected FALSE, can be several payment rows per order")
print("customers: customer_id unique?", dfs['customers']['customer_id'].is_unique)

orders: order_id unique?       True
order_items: (order_id) unique? False -> expected FALSE, many rows per order
order_payments: (order_id) unique? False -> expected FALSE, can be several payment rows per order
customers: customer_id unique? True


## Aggregate BEFORE you join
`order_items` and `order_payments` have **many rows per order**. If we join
them to `orders` directly we will duplicate order rows. So we aggregate each
of them down to one row per `order_id` first.

In [5]:
# ---- Aggregate order_items to one row per order ----
items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_unique_products=("product_id", "nunique"),
    n_unique_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
    max_shipping_limit_date=("shipping_limit_date", "max"),
).reset_index()

items_agg.head()

,order_id,n_items,n_unique_products,n_unique_sellers,total_price,total_freight,avg_item_price,max_shipping_limit_date
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90,2017-09-19 09:45:35
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90,2017-05-03 11:05:13
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00,2018-01-18 14:48:30
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99,2018-08-15 10:10:18
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90,2017-02-13 13:57:51


In [6]:
# ---- Aggregate order_payments to one row per order ----
payments_agg = order_payments.groupby("order_id").agg(
    n_payment_installments=("payment_installments", "max"),
    total_payment_value=("payment_value", "sum"),
    n_payment_rows=("payment_sequential", "count"),
).reset_index()

# most frequent payment type per order (simple, defensible choice)
payment_type_mode = (
    order_payments.groupby("order_id")["payment_type"]
    .agg(lambda s: s.value_counts().idxmax())
    .reset_index()
    .rename(columns={"payment_type": "main_payment_type"})
)
payments_agg = payments_agg.merge(payment_type_mode, on="order_id", how="left")
payments_agg.head()

,order_id,n_payment_installments,total_payment_value,n_payment_rows,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,2,72.19,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,3,259.83,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,5,216.87,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,2,25.78,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,3,218.04,1,credit_card


In [7]:
# ---- Bring in one product per order (dominant category), lightly ----
# We only need this at the light "join correctly" level here;
# deeper feature engineering happens in Notebook 5.
products = dfs["products"].merge(
    dfs["category_translation"], on="product_category_name", how="left"
)
item_products = order_items[["order_id", "product_id"]].merge(
    products[["product_id", "product_category_name_english"]],
    on="product_id", how="left"
)
main_category = (
    item_products.groupby("order_id")["product_category_name_english"]
    .agg(lambda s: s.value_counts().idxmax() if s.notna().any() else np.nan)
    .reset_index()
    .rename(columns={"product_category_name_english": "main_category"})
)
main_category.head()

,order_id,main_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools


In [8]:
# ---- Bring in customer state/city (one row per order via customer_id) ----
customer_geo = dfs["customers"][["customer_id", "customer_state", "customer_city",
                                   "customer_zip_code_prefix"]]

# ---- Final join: everything keyed on order_id, one row per order ----
ml_table = orders.merge(items_agg, on="order_id", how="left") \
                  .merge(payments_agg, on="order_id", how="left") \
                  .merge(main_category, on="order_id", how="left") \
                  .merge(customer_geo, on="customer_id", how="left")

print("ml_table shape:", ml_table.shape)
assert ml_table["order_id"].is_unique, "ml_table must be one row per order!"
ml_table.head()

ml_table shape: (99441, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,n_unique_products,...,avg_item_price,max_shipping_limit_date,n_payment_installments,total_payment_value,n_payment_rows,main_payment_type,main_category,customer_state,customer_city,customer_zip_code_prefix
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,1.0,...,29.99,2017-10-06 11:07:15,1.0,38.71,3.0,voucher,housewares,SP,sao paulo,3149
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,1.0,...,118.70,2018-07-30 03:24:27,1.0,141.46,1.0,boleto,perfumery,BA,barreiras,47813
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,1.0,...,159.90,2018-08-13 08:55:23,3.0,179.12,1.0,credit_card,auto,GO,vianopolis,75265
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,1.0,...,45.00,2017-11-23 19:45:59,1.0,72.20,1.0,credit_card,pet_shop,RN,sao goncalo do amarante,59296
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,1.0,...,19.90,2018-02-19 20:31:37,1.0,28.62,1.0,credit_card,stationery,SP,santo andre,9195


In [9]:
# ---- sanity checks before saving ----
print(ml_table.isna().mean().sort_values(ascending=False).head(10))
print("orders with no items info (order was likely never processed):",
      ml_table['n_items'].isna().sum())

order_delivered_customer_date    0.029817
main_category                    0.021973
order_delivered_carrier_date     0.017930
n_unique_products                0.007794
n_unique_sellers                 0.007794
n_items                          0.007794
avg_item_price                   0.007794
max_shipping_limit_date          0.007794
total_freight                    0.007794
total_price                      0.007794
dtype: float64
orders with no items info (order was likely never processed): 775


## Artifact: one ML table, one row per order
Saved for Notebook 2 to build the label on top of.

In [10]:
ml_table.to_parquet(f"{ARTIFACTS_DIR}/01_ml_table.parquet", index=False)
print("Saved:", f"{ARTIFACTS_DIR}/01_ml_table.parquet", "| shape:", ml_table.shape)

Saved: artifacts/01_ml_table.parquet | shape: (99441, 23)
